# Pipeline обработки данных.

Чтение очищенного датасета. (датасет чистим в `eda.ipynb`)

## Задание 1 предобработка представлен в `eda.ipynb`

In [2]:
import pandas as pd

In [3]:
dataset_path = "dataset_clean.csv"

df = pd.read_csv(dataset_path, parse_dates=["TS"])

df.head()

,TS,Lab1_G1_N1,Lab1_G1_N2,Lab1_G1_N3,Lab1_G1_P2,Lab1_G1_T4ср,Lab1_G1_T1,Lab1_G1_T607,Lab1_G1_T600,Lab1_G1_T638,...,Lab1_AVOM_AVOMN1,Lab1_Kp,Lab1_hGPA,Lab1_Kran_5,Lab1_Kran_2,Lab1_Kran_6,Lab1_U_Kran_GPA_A_APK,Lab1_Kran_1,Lab1_Kran_4,Lab1_q
0,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
1,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
2,2022-12-02 19:01:59.999999+00:00,8725,11493,5001,15.35,663.1,-16.0,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
3,2022-12-02 19:02:59.999999+00:00,8726,11493,5011,15.34,663.0,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
4,2022-12-02 19:04:00+00:00,8728,11496,5003,15.33,663.4,-15.9,36.1,67.1,80.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8


## Задание 2. Расчет показателей Хёрста и Ляпунова.

In [4]:
import numpy as np
import nolds

Подготовим вспомогательную функцию: ряд нормализуется. Константные ряды не анализируются, потому что для них R/S-анализ и показатель Ляпунова неинформативны.

In [5]:
def prepare_series(series):
    x = series.to_numpy()

    std = x.std()
    if std == 0:
        return None

    return (x - x.mean()) / std

Датасет является многоканальным временным рядом: в каждый момент времени измеряется несколько параметров установки. Поэтому на данном этапе каждый динамический числовой признак рассматривается как отдельный одномерный временной ряд.

Временная колонка `TS` используется только для упорядочивания наблюдений и не включается в расчёт показателей.

In [6]:
df_tda = df.sort_values("TS").reset_index(drop=True)

numeric_cols = df_tda.select_dtypes(include="number").columns.tolist()
numeric_cols = [
    col for col in numeric_cols
    if df_tda[col].nunique(dropna=False) > 1
]

print(f"Рядов для анализа: {len(numeric_cols)}")

Рядов для анализа: 84


Для каждого ряда считаем показатель Хёрста методом R/S-анализа и крупнейший показатель Ляпунова методом Розенштейна (`lyap_r`).

*пояснение*:

Для оценки показателя Ляпунова был использован метод Розенштейна, поскольку он позволяет оценить крупнейший показатель
Ляпунова непосредственно по одномерному временному ряду без задания аналитической модели системы. Метод основан на
реконструкции фазового пространства с помощью временных задержек и последующем анализе скорости расхождения близких
траекторий. Положительное значение крупнейшего показателя Ляпунова интерпретируется как признак чувствительности к
начальным условиям и возможной хаотической динамики, тогда как неположительное значение не подтверждает наличие
хаотического поведения. Такой подход является практичным для экспериментальных и промышленных временных рядов, где
доступны только наблюдения датчиков, но неизвестны уравнения порождающего процесса.

Для предварительной оценки показателя Ляпунова параметры lag и min_tsep подбирались встроенными эвристиками библиотеки nolds: задержка оценивается по автокорреляции, а минимальное временное разделение — по среднему периоду сигнала. На следующем этапе параметры вложения будут подбираться более подробно.

In [7]:
results = []

# Каждый числовой признак рассматриваем как отдельный одномерный временной ряд.
for col in numeric_cols:
    x = prepare_series(df_tda[col])

    # Если ряд константный, показатели для него не считаются.
    if x is None:
        results.append({
            "column": col,
            "hurst_rs": np.nan,
            "lyapunov": np.nan,
            "error": "constant series",
        })
        continue

    try:
        hurst = nolds.hurst_rs(
            x,
            fit="poly",
            corrected=True,
            unbiased=True,
        )
    except Exception as exc:
        hurst = np.nan
        hurst_error = str(exc)
    else:
        hurst_error = ""

    try:
        lyapunov = nolds.lyap_r(
            x,
            lag=1,
            fit="poly",
        )
    except Exception as exc:
        lyapunov = np.nan
        lyapunov_error = str(exc)
    else:
        lyapunov_error = ""

    # Сохраняем численные значения и возможные ошибки расчета для диагностики.
    results.append({
        "column": col,
        "hurst_rs": hurst,
        "lyapunov": lyapunov,
        "error": "; ".join(
            err for err in [hurst_error, lyapunov_error]
            if err
        ),
    })

/home/smokehappiest/tda-2026/.venv/lib/python3.12/site-packages/nolds/measures.py:263: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 359
  warnings.warn(msg.format(min_tsep), RuntimeWarning)


| Колонка    | Пояснение                                                                                                   |
| ---------- | ----------------------------------------------------------------------------------------------------------- |
| `column`   | Название исследуемого признака (временного ряда / сенсора / параметра установки)                            |
| `hurst_rs` | Показатель Херста, характеризующий наличие долгосрочной памяти и степень персистентности временного ряда    |
| `lyapunov` | Оценка наибольшего показателя Ляпунова, характеризующая чувствительность динамики ряда к начальным условиям |
| `error`    | Текст ошибки вычисления метрик (если расчёт для признака завершился неуспешно)                              |


In [8]:
process_report = pd.DataFrame(results)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
process_report

,column,hurst_rs,lyapunov,error
0,Lab1_G1_N1,0.896420,0.066899,
1,Lab1_G1_N2,0.892538,0.078063,
2,Lab1_G1_N3,0.485665,0.042443,
3,Lab1_G1_P2,0.744458,0.065301,
4,Lab1_G1_T4ср,0.885607,0.077858,
5,Lab1_G1_T1,0.902912,0.070435,
6,Lab1_G1_T607,0.723786,0.029808,
7,Lab1_G1_T600,0.906647,0.070207,
8,Lab1_G1_T638,0.937976,0.071134,
9,Lab1_G1_T606,0.883686,0.069799,


Добавим качественную интерпретацию.

| Термин | Семантически | Показатель Хёрста |
|---|---|---|
| **Статистически устойчивый / стационарный процесс** | По конспекту: статистические характеристики процесса устойчивы во времени | $$H < 0$$ или $$H > 1$$ |
| **Антиперсистентный процесс** | Ряд часто меняет направление: после роста вероятен спад, после спада — рост | $$0 \leq H \leq 0.4$$ |
| **Слабая антиперсистентность, возможна хаотическая динамика** | Возвратное поведение выражено слабо; возможны элементы сложной динамики | $$0.4 < H \leq 0.5$$ |
| **Слабая персистентность, возможна хаотическая динамика** | Долговременная память выражена слабо; возможны элементы сложной динамики | $$0.5 < H \leq 0.6$$ |
| **Персистентный процесс** | Ряд “помнит” прошлое: если рос, скорее продолжит расти; если снижался — продолжит снижаться | $$0.6 < H \leq 1$$ |


In [9]:
h = process_report["hurst_rs"]

process_report["hurst_type"] = np.select(
    [
        h.isna(),
        h < 0,
        h <= 0.4,
        (h > 0.4) & (h <= 0.5),
        (h > 0.5) & (h <= 0.6),
        (h > 0.6) & (h <= 1.0),
        h > 1.0,
    ],
    [
        "H < 0: статистически устойчивый / стационарный процесс",
        "значение H вне ожидаемого диапазона",
        "антиперсистентный процесс",
        "слабая антиперсистентность, возможна хаотическая динамика",
        "слабая персистентность, возможна хаотическая динамика",
        "персистентный процесс",
        "H > 1: статистически устойчивый / стационарный процесс",
    ],
    default="не определено",
)

process_report["lyapunov_type"] = np.select(
    [
        process_report["lyapunov"] > 0,
        process_report["lyapunov"] <= 0,
    ],
    [
        "признаки хаотической динамики",
        "хаотичность не подтверждается",
    ],
    default="не определено",
)

display(process_report.sort_values("lyapunov", ascending=False))

,column,hurst_rs,lyapunov,error,hurst_type,lyapunov_type
44,Lab1_G3_Pc1,0.441834,0.160560,,"слабая антиперсистентность, возможна хаотическ...",признаки хаотической динамики
45,Lab1_G3_Pc2,0.694835,0.123888,,персистентный процесс,признаки хаотической динамики
46,Lab1_G3_Pc3,0.556710,0.111137,,"слабая персистентность, возможна хаотическая д...",признаки хаотической динамики
55,Lab1_G3_ЗО_СТ,0.464293,0.110202,,"слабая антиперсистентность, возможна хаотическ...",признаки хаотической динамики
54,Lab1_G3_ПО_СТ,1.078973,0.103721,,H > 1: статистически устойчивый / стационарный...,признаки хаотической динамики
38,Lab1_G3_Lm,0.722304,0.101253,,персистентный процесс,признаки хаотической динамики
52,Lab1_G3_КВД,0.701866,0.100767,,персистентный процесс,признаки хаотической динамики
51,Lab1_G3_КНД,0.701866,0.100767,,персистентный процесс,признаки хаотической динамики
41,Lab1_G3_T638,0.903007,0.090695,,персистентный процесс,признаки хаотической динамики
47,Lab1_G3_T600,0.909729,0.089369,,персистентный процесс,признаки хаотической динамики


Сводные статистики нужны для общего вывода о природе порождающих процессов по всем временным рядам.

In [10]:
display(process_report[["hurst_rs", "lyapunov"]].describe())
display(process_report["hurst_type"].value_counts().to_frame("count"))
display(process_report["lyapunov_type"].value_counts().to_frame("count"))

,hurst_rs,lyapunov
count,84.000000,84.000000
mean,0.744078,0.057975
std,0.175706,0.025162
min,0.388588,0.018189
25%,0.606125,0.041532
50%,0.731952,0.049385
75%,0.874413,0.071383
max,1.362306,0.160560


,count
hurst_type,
персистентный процесс,60
"слабая персистентность, возможна хаотическая динамика",12
"слабая антиперсистентность, возможна хаотическая динамика",6
H > 1: статистически устойчивый / стационарный процесс,5
антиперсистентный процесс,1


,count
lyapunov_type,
признаки хаотической динамики,84


### Выводы по показателям Хёрста и Ляпунова



### Интерпретация результатов перед вложением в облако точек


## Проверка наличия линейной лаговой связи

In [11]:
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr

try:
    from sklearn.metrics import normalized_mutual_info_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("sklearn не найден: взаимная информация считаться не будет")

In [12]:
def discretize_by_quantiles(x, n_bins=10):
    """
    Дискретизация значений по квантилям.
    Нужна для расчета normalized mutual information.
    """
    x = pd.Series(x)

    # Если уникальных значений мало, уменьшаем число бинов
    unique_count = x.nunique()
    bins = min(n_bins, unique_count)

    if bins < 2:
        return None

    try:
        return pd.qcut(x, q=bins, labels=False, duplicates="drop")
    except ValueError:
        return None

In [13]:
def analyze_lag_linearity_for_series(x, column, max_lag=50, n_bins=10):
    """
    Анализ связи между x(t) и x(t + tau) для одного временного ряда.

    Pearson  — линейная зависимость.
    Spearman — монотонная зависимость.
    NMI      — общая зависимость, в том числе нелинейная.
    """
    rows = []

    max_lag = min(max_lag, len(x) // 3)

    for tau in range(1, max_lag + 1):
        x_t = x[:-tau]
        x_lag = x[tau:]

        # Если после сдвига данных слишком мало — пропускаем
        if len(x_t) < 30:
            continue

        # Pearson: линейная связь
        try:
            pearson_corr, pearson_p = pearsonr(x_t, x_lag)
        except Exception:
            pearson_corr, pearson_p = np.nan, np.nan

        # Spearman: монотонная связь
        try:
            spearman_corr, spearman_p = spearmanr(x_t, x_lag)
        except Exception:
            spearman_corr, spearman_p = np.nan, np.nan

        # Normalized Mutual Information: более общий тип зависимости
        if SKLEARN_AVAILABLE:
            x_t_disc = discretize_by_quantiles(x_t, n_bins=n_bins)
            x_lag_disc = discretize_by_quantiles(x_lag, n_bins=n_bins)

            if x_t_disc is not None and x_lag_disc is not None:
                nmi = normalized_mutual_info_score(x_t_disc, x_lag_disc)
            else:
                nmi = np.nan
        else:
            nmi = np.nan

        rows.append({
            "column": column,
            "tau": tau,
            "pearson_corr": pearson_corr,
            "pearson_abs": abs(pearson_corr) if not np.isnan(pearson_corr) else np.nan,
            "pearson_p": pearson_p,
            "spearman_corr": spearman_corr,
            "spearman_abs": abs(spearman_corr) if not np.isnan(spearman_corr) else np.nan,
            "spearman_p": spearman_p,
            "nmi": nmi,
        })

    return rows

In [14]:
lag_results = []

for col in numeric_cols:
    x = prepare_series(df_tda[col])

    if x is None:
        continue

    rows = analyze_lag_linearity_for_series(
        x=x,
        column=col,
        max_lag=50,
        n_bins=10,
    )

    lag_results.extend(rows)

lag_linearity_report = pd.DataFrame(lag_results)

display(lag_linearity_report.head())
print("Всего строк отчета:", len(lag_linearity_report))

,column,tau,pearson_corr,pearson_abs,pearson_p,spearman_corr,spearman_abs,spearman_p,nmi
0,Lab1_G1_N1,1,0.990183,0.990183,0.0,0.983904,0.983904,0.0,0.624119
1,Lab1_G1_N1,2,0.981837,0.981837,0.0,0.972542,0.972542,0.0,0.554717
2,Lab1_G1_N1,3,0.972523,0.972523,0.0,0.960188,0.960188,0.0,0.498699
3,Lab1_G1_N1,4,0.964356,0.964356,0.0,0.950374,0.950374,0.0,0.470247
4,Lab1_G1_N1,5,0.956212,0.956212,0.0,0.941499,0.941499,0.0,0.440945


Всего строк отчета: 4200


In [15]:
def summarize_lag_linearity(group):
    """
    Сводка по одному временному ряду.
    """
    best_pearson_row = group.loc[group["pearson_abs"].idxmax()]
    best_spearman_row = group.loc[group["spearman_abs"].idxmax()]

    if group["nmi"].notna().any():
        best_nmi_row = group.loc[group["nmi"].idxmax()]
        max_nmi = best_nmi_row["nmi"]
        tau_max_nmi = best_nmi_row["tau"]
        pearson_at_max_nmi = best_nmi_row["pearson_corr"]
    else:
        max_nmi = np.nan
        tau_max_nmi = np.nan
        pearson_at_max_nmi = np.nan

    return pd.Series({
        "max_abs_pearson": best_pearson_row["pearson_abs"],
        "tau_max_pearson": best_pearson_row["tau"],
        "pearson_at_best_tau": best_pearson_row["pearson_corr"],

        "max_abs_spearman": best_spearman_row["spearman_abs"],
        "tau_max_spearman": best_spearman_row["tau"],
        "spearman_at_best_tau": best_spearman_row["spearman_corr"],

        "max_nmi": max_nmi,
        "tau_max_nmi": tau_max_nmi,
        "pearson_at_max_nmi": pearson_at_max_nmi,

        "mean_abs_pearson": group["pearson_abs"].mean(),
        "mean_abs_spearman": group["spearman_abs"].mean(),
        "mean_nmi": group["nmi"].mean(),
    })


linearity_summary = (
    lag_linearity_report
    .groupby("column")
    .apply(summarize_lag_linearity)
    .reset_index()
)

display(linearity_summary.head())

,column,max_abs_pearson,tau_max_pearson,pearson_at_best_tau,max_abs_spearman,tau_max_spearman,spearman_at_best_tau,max_nmi,tau_max_nmi,pearson_at_max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi
0,Lab1_G1_N1,0.990183,1.0,0.990183,0.983904,1.0,0.983904,0.624119,1.0,0.990183,0.774715,0.818693,0.284817
1,Lab1_G1_N2,0.994609,1.0,0.994609,0.989325,1.0,0.989325,0.686090,1.0,0.994609,0.777170,0.850767,0.342443
2,Lab1_G1_N3,0.148135,1.0,0.148135,0.136704,1.0,0.136704,0.025873,1.0,0.148135,0.030118,0.028786,0.013948
3,Lab1_G1_P2,0.864980,1.0,0.864980,0.915026,1.0,0.915026,0.461873,1.0,0.864980,0.500484,0.725153,0.250214
4,Lab1_G1_T1,0.993162,1.0,0.993162,0.989900,1.0,0.989900,0.696678,1.0,0.993162,0.831260,0.870121,0.365977


In [16]:
def classify_linear_dependence(max_abs_pearson):
    """
    Качественная интерпретация линейной зависимости по максимальной корреляции Пирсона.
    Пороги эвристические, нужны для EDA.
    """
    if pd.isna(max_abs_pearson):
        return "не определено"
    elif max_abs_pearson >= 0.7:
        return "сильная линейная лаговая связь"
    elif max_abs_pearson >= 0.3:
        return "умеренная линейная лаговая связь"
    else:
        return "слабая линейная лаговая связь"


def classify_possible_nonlinearity(row):
    """
    Эвристика:
    если общая зависимость по NMI заметная,
    но линейная корреляция при этом слабая,
    то связь может быть нелинейной.
    """
    if pd.isna(row["max_nmi"]):
        return "не оценивалось"

    if row["max_nmi"] >= 0.20 and abs(row["pearson_at_max_nmi"]) < 0.30:
        return "возможна нелинейная зависимость"
    elif row["max_nmi"] >= 0.20 and abs(row["pearson_at_max_nmi"]) >= 0.30:
        return "есть зависимость, частично линейная"
    elif row["max_nmi"] < 0.20:
        return "слабая общая зависимость"
    else:
        return "не определено"


linearity_summary["linear_dependence_type"] = linearity_summary["max_abs_pearson"].apply(
    classify_linear_dependence
)

linearity_summary["nonlinear_dependence_type"] = linearity_summary.apply(
    classify_possible_nonlinearity,
    axis=1
)

display(linearity_summary)

,column,max_abs_pearson,tau_max_pearson,pearson_at_best_tau,max_abs_spearman,tau_max_spearman,spearman_at_best_tau,max_nmi,tau_max_nmi,pearson_at_max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi,linear_dependence_type,nonlinear_dependence_type
0,Lab1_G1_N1,0.990183,1.0,0.990183,0.983904,1.0,0.983904,0.624119,1.0,0.990183,0.774715,0.818693,0.284817,сильная линейная лаговая связь,"есть зависимость, частично линейная"
1,Lab1_G1_N2,0.994609,1.0,0.994609,0.989325,1.0,0.989325,0.686090,1.0,0.994609,0.777170,0.850767,0.342443,сильная линейная лаговая связь,"есть зависимость, частично линейная"
2,Lab1_G1_N3,0.148135,1.0,0.148135,0.136704,1.0,0.136704,0.025873,1.0,0.148135,0.030118,0.028786,0.013948,слабая линейная лаговая связь,слабая общая зависимость
3,Lab1_G1_P2,0.864980,1.0,0.864980,0.915026,1.0,0.915026,0.461873,1.0,0.864980,0.500484,0.725153,0.250214,сильная линейная лаговая связь,"есть зависимость, частично линейная"
4,Lab1_G1_T1,0.993162,1.0,0.993162,0.989900,1.0,0.989900,0.696678,1.0,0.993162,0.831260,0.870121,0.365977,сильная линейная лаговая связь,"есть зависимость, частично линейная"
5,Lab1_G1_T1002,0.994915,1.0,0.994915,0.992165,1.0,0.992165,0.739825,1.0,0.994915,0.839411,0.882837,0.393626,сильная линейная лаговая связь,"есть зависимость, частично линейная"
6,Lab1_G1_T1003,0.961275,1.0,0.961275,0.931186,1.0,0.931186,0.467518,1.0,0.961275,0.799384,0.732945,0.241009,сильная линейная лаговая связь,"есть зависимость, частично линейная"
7,Lab1_G1_T4ср,0.993846,1.0,0.993846,0.988979,1.0,0.988979,0.684159,1.0,0.993846,0.754363,0.850433,0.347538,сильная линейная лаговая связь,"есть зависимость, частично линейная"
8,Lab1_G1_T600,0.996354,1.0,0.996354,0.995529,1.0,0.995529,0.844807,1.0,0.996354,0.902270,0.903053,0.439552,сильная линейная лаговая связь,"есть зависимость, частично линейная"
9,Lab1_G1_T606,0.988727,1.0,0.988727,0.984977,1.0,0.984977,0.628262,1.0,0.988727,0.776923,0.821335,0.298091,сильная линейная лаговая связь,"есть зависимость, частично линейная"


In [17]:
display(linearity_summary["linear_dependence_type"].value_counts().to_frame("count"))

display(linearity_summary["nonlinear_dependence_type"].value_counts().to_frame("count"))

display(
    linearity_summary[
        [
            "max_abs_pearson",
            "max_abs_spearman",
            "max_nmi",
            "mean_abs_pearson",
            "mean_abs_spearman",
            "mean_nmi",
        ]
    ].describe()
)

,count
linear_dependence_type,
сильная линейная лаговая связь,50
слабая линейная лаговая связь,19
умеренная линейная лаговая связь,15


,count
nonlinear_dependence_type,
"есть зависимость, частично линейная",54
слабая общая зависимость,28
возможна нелинейная зависимость,2


,max_abs_pearson,max_abs_spearman,max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi
count,84.000000,84.000000,84.000000,84.000000,84.000000,84.000000
mean,0.675867,0.659745,0.427082,0.462470,0.479817,0.268736
std,0.339527,0.338861,0.321139,0.330967,0.341040,0.271317
min,0.063260,0.063070,0.008583,0.017758,0.019170,0.002955
25%,0.387973,0.335088,0.073073,0.123201,0.114310,0.030706
50%,0.850462,0.820613,0.465244,0.447666,0.486961,0.244922
75%,0.977479,0.973372,0.676757,0.777170,0.819353,0.370776
max,0.999673,0.996839,1.000000,0.996013,0.979670,1.000000


# Задание 3
---
## Вложение временного ряда
На основе выводов из п.2 преобразуйте одномерный временной ряд в многомерное облако точек.
- Равномерное вложение (Uniform Embedding):
  - Определите оптимальные временную задержку и размерность вложения.
- Неравномерное вложение (Non-uniform Embedding):
  - Определите оптимальные различные (неравномерные) шаги задержки (размерность определяется одновременно).

Анализ: Визуализируйте полученные облака точек (в 2D или 3D проекциях, например, через PCA/UMAP). Сравните структуру аттракторов, полученных равномерным и неравномерным методами.


In [18]:
import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances

### Функция равномерного вложения

Базовая формула равномерного вложения временного ряда имеет вид:

$$
Y_i = \left(
x_i,\;
x_{i+\tau},\;
x_{i+2\tau},\;
\dots,\;
x_{i+(m-1)\tau}
\right)
$$

In [19]:
def delay_embedding(x, m, tau):
    """
    Равномерное вложение временного ряда с задержкой.

    x   — одномерный временной ряд;
    m   — размерность вложения;
    tau — временная задержка.
    """
    x = np.asarray(x)

    n_points = len(x) - (m - 1) * tau

    if n_points <= 0:
        raise ValueError("Ряд слишком короткий для выбранных m и tau")

    embedded = np.array([
        [x[i + j * tau] for j in range(m)]
        for i in range(n_points)
    ])

    return embedded

## Метод усиления шума для оценки качества вложения

In [20]:
def noise_amplification_score(
    x,
    m,
    tau,
    k_neighbors=10,
    prediction_horizon=1,
    theiler_window=None,
    max_points=1200,
    random_state=42,
):
    """
    Упрощенная практическая оценка усиления шума.

    Идея:
    если вложение хорошее, то близкие точки фазового пространства
    должны иметь похожее будущее.

    Возвращает score: чем меньше, тем лучше.
    """
    x = np.asarray(x)

    if theiler_window is None:
        theiler_window = tau

    # Строим вложение
    X = delay_embedding(x, m=m, tau=tau)

    # Для каждой точки берем будущее значение после prediction_horizon
    future_start = (m - 1) * tau + prediction_horizon
    valid_n = len(x) - future_start

    if valid_n <= k_neighbors + 2:
        return np.nan

    X = X[:valid_n]
    y_future = x[future_start:future_start + valid_n]

    # Чтобы не взорваться по памяти на больших рядах, можно взять подвыборку
    if len(X) > max_points:
        rng = np.random.default_rng(random_state)
        idx = np.sort(rng.choice(len(X), size=max_points, replace=False))
        X = X[idx]
        y_future = y_future[idx]

    n = len(X)

    if n <= k_neighbors + 2:
        return np.nan

    # Матрица расстояний между точками фазового пространства
    D = pairwise_distances(X)

    # Исключаем саму точку и слишком близкие по времени точки
    indices = np.arange(n)
    time_distance = np.abs(indices[:, None] - indices[None, :])
    D[time_distance <= theiler_window] = np.inf

    local_scores = []

    for i in range(n):
        neighbor_idx = np.argsort(D[i])[:k_neighbors]

        # Если соседей не хватает
        if np.isinf(D[i, neighbor_idx]).any():
            continue

        local_radius = np.mean(D[i, neighbor_idx])

        if local_radius <= 1e-12:
            continue

        # Насколько отличаются будущие значения у ближайших соседей
        future_std = np.std(y_future[neighbor_idx])

        # Нормируем разброс будущего на размер локальной окрестности
        local_score = future_std / local_radius
        local_scores.append(local_score)

    if len(local_scores) == 0:
        return np.nan

    # Логарифмируем, чтобы сгладить выбросы
    return np.mean(np.log1p(local_scores))

## Перебор m и tau

In [21]:
def uniform_embedding_grid_search(
    x,
    m_values=range(2, 7),
    tau_values=range(1, 51),
    k_neighbors=10,
    prediction_horizon=1,
):
    """
    Перебирает пары (m, tau) и выбирает ту, где усиление шума минимально.
    """
    rows = []

    for m in m_values:
        for tau in tau_values:
            # Проверка, что после вложения останется достаточно точек
            n_embedded = len(x) - (m - 1) * tau

            if n_embedded < 50:
                continue

            score = noise_amplification_score(
                x=x,
                m=m,
                tau=tau,
                k_neighbors=k_neighbors,
                prediction_horizon=prediction_horizon,
            )

            rows.append({
                "m": m,
                "tau": tau,
                "n_points": n_embedded,
                "noise_score": score,
            })

    result = pd.DataFrame(rows)

    if result.empty:
        return result, None

    result = result.sort_values("noise_score", ascending=True)
    best = result.iloc[0].to_dict()

    return result, best

## Выбор ряда для первого вложения

In [22]:
candidate_report = process_report.copy()

# Если есть отчет по линейности, присоединим его
if "linearity_summary" in globals():
    candidate_report = candidate_report.merge(
        linearity_summary,
        on="column",
        how="left"
    )

# Берем хорошие кандидаты для вложения
candidates = candidate_report[
    (candidate_report["hurst_rs"] > 0.6) &
    (candidate_report["lyapunov"] > 0)
].copy()

# Если есть max_abs_pearson, используем его тоже
if "max_abs_pearson" in candidates.columns:
    candidates = candidates.sort_values(
        by=["max_abs_pearson", "hurst_rs", "lyapunov"],
        ascending=False
    )
else:
    candidates = candidates.sort_values(
        by=["hurst_rs", "lyapunov"],
        ascending=False
    )

display(candidates.head(60))

,column,hurst_rs,lyapunov,error,hurst_type,lyapunov_type,max_abs_pearson,tau_max_pearson,pearson_at_best_tau,max_abs_spearman,tau_max_spearman,spearman_at_best_tau,max_nmi,tau_max_nmi,pearson_at_max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi,linear_dependence_type,nonlinear_dependence_type
78,Lab1_PposleNag,1.024337,0.028849,,H > 1: статистически устойчивый / стационарный...,признаки хаотической динамики,0.999673,1.0,0.999673,0.996056,1.0,0.996056,0.794901,1.0,0.999673,0.996013,0.979670,0.678196,сильная линейная лаговая связь,"есть зависимость, частично линейная"
80,Lab1_PdoNag,1.037843,0.037352,,H > 1: статистически устойчивый / стационарный...,признаки хаотической динамики,0.999434,1.0,0.999434,0.995801,1.0,0.995801,0.748796,1.0,0.999434,0.993427,0.976376,0.623669,сильная линейная лаговая связь,"есть зависимость, частично линейная"
70,Lab1_TC_Tm,0.816101,0.060596,,персистентный процесс,признаки хаотической динамики,0.997749,1.0,0.997749,0.996839,1.0,0.996839,0.877257,1.0,0.997749,0.886642,0.906901,0.481343,сильная линейная лаговая связь,"есть зависимость, частично линейная"
7,Lab1_G1_T600,0.906647,0.070207,,персистентный процесс,признаки хаотической динамики,0.996354,1.0,0.996354,0.995529,1.0,0.995529,0.844807,1.0,0.996354,0.902270,0.903053,0.439552,сильная линейная лаговая связь,"есть зависимость, частично линейная"
58,Lab1_G4_N1пр,0.953044,0.027054,,персистентный процесс,признаки хаотической динамики,0.996059,1.0,0.996059,0.979303,1.0,0.979303,0.637269,1.0,0.996059,0.978872,0.945159,0.505531,сильная линейная лаговая связь,"есть зависимость, частично линейная"
10,Lab1_G1_T1002,0.932611,0.072182,,персистентный процесс,признаки хаотической динамики,0.994915,1.0,0.994915,0.992165,1.0,0.992165,0.739825,1.0,0.994915,0.839411,0.882837,0.393626,сильная линейная лаговая связь,"есть зависимость, частично линейная"
1,Lab1_G1_N2,0.892538,0.078063,,персистентный процесс,признаки хаотической динамики,0.994609,1.0,0.994609,0.989325,1.0,0.989325,0.686090,1.0,0.994609,0.777170,0.850767,0.342443,сильная линейная лаговая связь,"есть зависимость, частично линейная"
60,Lab1_G4_N2,0.892538,0.078063,,персистентный процесс,признаки хаотической динамики,0.994609,1.0,0.994609,0.989325,1.0,0.989325,0.686090,1.0,0.994609,0.777170,0.850767,0.342443,сильная линейная лаговая связь,"есть зависимость, частично линейная"
8,Lab1_G1_T638,0.937976,0.071134,,персистентный процесс,признаки хаотической динамики,0.994587,1.0,0.994587,0.992895,1.0,0.992895,0.770445,1.0,0.994587,0.860274,0.876135,0.394296,сильная линейная лаговая связь,"есть зависимость, частично линейная"
4,Lab1_G1_T4ср,0.885607,0.077858,,персистентный процесс,признаки хаотической динамики,0.993846,1.0,0.993846,0.988979,1.0,0.988979,0.684159,1.0,0.993846,0.754363,0.850433,0.347538,сильная линейная лаговая связь,"есть зависимость, частично линейная"


In [23]:
selected_col = candidates.iloc[57]["column"]

print("Выбранный ряд:", selected_col)

x = prepare_series(df_tda[selected_col])

print("Длина ряда:", len(x))
print("Среднее после нормировки:", x.mean())
print("Стандартное отклонение после нормировки:", x.std())

Выбранный ряд: Lab1_G2_F3
Длина ряда: 1439
Среднее после нормировки: 2.370121703716804e-16
Стандартное отклонение после нормировки: 0.9999999999999999


In [24]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


def delay_embedding(x, m, tau):
    """
    Задержанное вложение временного ряда.

    Для m=3 и tau=70 строит точки:
    (x[t], x[t + 70], x[t + 140])
    """
    x = np.asarray(x)

    n_points = len(x) - (m - 1) * tau

    if n_points <= 0:
        raise ValueError(
            "Ряд слишком короткий для выбранных параметров "
            f"m={m}, tau={tau}. Нужно больше {(m - 1) * tau} наблюдений."
        )

    embedded = np.column_stack([
        x[i * tau : i * tau + n_points]
        for i in range(m)
    ])

    return embedded

In [25]:
# Параметры для вложения одного ряда!
TAU_SINGLE=95
M_DIMENSION=3

In [28]:
# Берем выбранный ряд
x = df_tda[selected_col].dropna().values

# Строим задержанное вложение
X_embedded = delay_embedding(x, m=M_DIMENSION, tau=TAU_SINGLE)

print("Выбранный ряд:", selected_col)
print("Размер исходного ряда:", len(x))
print("Размер облака после вложения:", X_embedded.shape)

embedding_df = pd.DataFrame(
    X_embedded,
    columns=[f"x(t + {i * TAU_SINGLE})" if i > 0 else "x(t)" for i in range(M_DIMENSION)]
)

display(embedding_df.head())

Выбранный ряд: Lab1_G2_F3
Размер исходного ряда: 1439
Размер облака после вложения: (1249, 3)


,x(t),x(t + 95),x(t + 190)
0,1.23,1.35,1.30
1,1.23,1.32,1.34
2,1.08,1.36,1.36
3,1.18,1.37,1.43
4,1.03,1.23,1.38


In [30]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_embedded[:, 0],
        y=X_embedded[:, 1],
        z=X_embedded[:, 2],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.7
        )
    )
)

fig.update_layout(
    title=f"Задержанное вложение ряда {selected_col}: m={M_DIMENSION}, tau={TAU_SINGLE}",
    scene=dict(
        xaxis_title="x(t)",
        yaxis_title=f"x(t + {TAU_SINGLE})",
        zaxis_title=f"x(t + {2 * TAU_SINGLE})"
    ),
    width=900,
    height=700
)

fig.show()

In [32]:
import itertools
import numpy as np
import plotly.graph_objects as go

# Все возможные 3D-проекции из 5 координат
projection_indices = list(itertools.combinations(range(M_DIMENSION), 3))

print("Доступные 3D-проекции:")
for idx, comb in enumerate(projection_indices):
    labels = [embedding_df.columns[i] for i in comb]
    print(f"{idx}: {labels}")

Доступные 3D-проекции:
0: ['x(t)', 'x(t + 95)', 'x(t + 190)']


In [37]:
# Номер проекции из списка выше
projection_id = 0

i, j, k = projection_indices[projection_id]

time_index = np.arange(X_embedded.shape[0])

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X_embedded[:, i],
        y=X_embedded[:, j],
        z=X_embedded[:, k],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.75,
            color=time_index,
            colorscale="Viridis",
            colorbar=dict(title="t")
        )
    )
)

fig.update_layout(
    title=(
        f"3D-проекция задержанного вложения ряда {selected_col}<br>"
        f"m={M_DIMENSION}, tau={TAU_SINGLE}: "
        f"{embedding_df.columns[i]}, {embedding_df.columns[j]}, {embedding_df.columns[k]}"
    ),
    scene=dict(
        xaxis_title=embedding_df.columns[i],
        yaxis_title=embedding_df.columns[j],
        zaxis_title=embedding_df.columns[k]
    ),
    width=900,
    height=700
)

fig.show()

# Попытка вложить много временных рядов

In [38]:
def delay_embedding(series, m: int, tau: int) -> np.ndarray:
    """
    Равномерное лаговое вложение временного ряда.

    Преобразует одномерный ряд x(t) в облако точек:

        X(t) = [x(t), x(t + tau), x(t + 2*tau), ..., x(t + (m-1)*tau)]

    Parameters
    ----------
    series : array-like
        Одномерный временной ряд.
    m : int
        Размерность вложения.
    tau : int
        Временная задержка.

    Returns
    -------
    np.ndarray
        Матрица размера (N - (m - 1) * tau, m),
        где каждая строка — точка фазового пространства.
    """

    if m < 2:
        raise ValueError("m должно быть >= 2")

    if tau < 1:
        raise ValueError("tau должно быть >= 1")

    x = np.asarray(series, dtype=float)
    x = x[~np.isnan(x)]

    n_points = len(x) - (m - 1) * tau

    if n_points <= 0:
        raise ValueError(
            f"Слишком большие m={m} и tau={tau} для ряда длины {len(x)}"
        )

    embedded = np.column_stack([
        x[i * tau : i * tau + n_points]
        for i in range(m)
    ])

    return embedded

In [39]:
from pathlib import Path

BEST_CHOICES_PATH = "uzal_best_choices_by_column.csv"
if not Path(BEST_CHOICES_PATH).exists():
    BEST_CHOICES_PATH = "uzal_cost_final_result.csv"

NORMALIZE_SERIES = True

best_choices = pd.read_csv(BEST_CHOICES_PATH)

required_columns = {"column", "tau", "dimension", "uzal_cost"}
missing_columns = required_columns - set(best_choices.columns)
if missing_columns:
    raise ValueError(f"В {BEST_CHOICES_PATH} не хватает колонок: {sorted(missing_columns)}")

all_embeddings = {}
embedding_rows = []
embedding_errors = []

for row in best_choices.itertuples(index=False):
    col = row.column
    tau = int(row.tau)
    m = int(row.dimension)

    if col not in df_tda.columns:
        embedding_errors.append({
            "column": col,
            "tau": tau,
            "dimension": m,
            "error": "нет такой колонки в df_tda",
        })
        continue

    series = pd.to_numeric(df_tda[col], errors="coerce").dropna()

    if NORMALIZE_SERIES:
        std = series.std(ddof=0)
        if std == 0 or pd.isna(std):
            embedding_errors.append({
                "column": col,
                "tau": tau,
                "dimension": m,
                "error": "константный ряд",
            })
            continue
        series = (series - series.mean()) / std

    try:
        X = delay_embedding(series.to_numpy(), m=m, tau=tau)
    except Exception as exc:
        embedding_errors.append({
            "column": col,
            "tau": tau,
            "dimension": m,
            "error": str(exc),
        })
        continue

    all_embeddings[col] = {
        "X": X,
        "tau": tau,
        "dimension": m,
        "uzal_cost": float(row.uzal_cost),
    }

    embedding_rows.append({
        "column": col,
        "tau": tau,
        "dimension": m,
        "uzal_cost": float(row.uzal_cost),
        "series_length": len(series),
        "embedding_points": X.shape[0],
        "embedding_dimension": X.shape[1],
    })

embedding_summary = pd.DataFrame(embedding_rows).sort_values("uzal_cost").reset_index(drop=True)
embedding_errors = pd.DataFrame(embedding_errors)

print(f"Построено вложений: {len(all_embeddings)}")
print(f"Ошибок: {len(embedding_errors)}")

display(embedding_summary.head(20))
if len(embedding_errors) > 0:
    display(embedding_errors)

Построено вложений: 74
Ошибок: 0


,column,tau,dimension,uzal_cost,series_length,embedding_points,embedding_dimension
0,Lab1_G3_Lm,30,3,-3.305123,1439,1379,3
1,Lab1_Rc,80,11,-2.885113,1439,639,11
2,Lab1_Hpol,69,5,-2.501856,1439,1163,5
3,Lab1_he,23,8,-2.348359,1439,1278,8
4,Lab1_dev,9,4,-2.216724,1439,1412,4
5,Lab1_G3_T638,13,3,-1.933533,1439,1413,3
6,Lab1_G3_T600,20,3,-1.916388,1439,1399,3
7,Lab1_PdoNag,79,5,-1.823454,1439,1123,5
8,Lab1_Pm_sm_N,96,10,-1.807554,1439,575,10
9,Lab1_dPmg,88,9,-1.807238,1439,735,9


In [ ]:
selected_embedding_col = embedding_summary.iloc[0]["column"]

embedding_info = all_embeddings[selected_embedding_col]
X = embedding_info["X"]
tau = embedding_info["tau"]
m = embedding_info["dimension"]

time_index = np.arange(X.shape[0])

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=X[:, 0],
        y=X[:, 1],
        z=X[:, 2],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.75,
            color=time_index,
            colorscale="Viridis",
            colorbar=dict(title="t"),
        ),
    )
)

fig.update_layout(
    title=f"Задержанное вложение ряда {selected_embedding_col}: m={m}, tau={tau}",
    scene=dict(
        xaxis_title="x(t)",
        yaxis_title=f"x(t + {tau})",
        zaxis_title=f"x(t + {2 * tau})",
    ),
    width=900,
    height=700,
)

fig.show()

In [41]:
import numpy as np
import plotly.graph_objects as go

# ограничим число точек, чтобы график не лагал
MAX_POINTS = 3000

fig = go.Figure()

columns = list(all_embeddings.keys())

for i, col in enumerate(columns):
    info = all_embeddings[col]
    X = info["X"]
    tau = info["tau"]
    m = info["dimension"]

    # если размерность больше 3 — берём первые 3 координаты
    if X.shape[1] < 3:
        continue

    X_plot = X[:MAX_POINTS]
    time_index = np.arange(X_plot.shape[0])

    fig.add_trace(
        go.Scatter3d(
            x=X_plot[:, 0],
            y=X_plot[:, 1],
            z=X_plot[:, 2],
            mode="markers",
            marker=dict(
                size=3,
                opacity=0.75,
                color=time_index,
                colorscale="Viridis",
                colorbar=dict(title="t") if i == 0 else None,
            ),
            name=f"{col}: m={m}, tau={tau}",
            visible=(i == 0),
        )
    )

buttons = []

for i, col in enumerate(columns):
    info = all_embeddings[col]
    tau = info["tau"]
    m = info["dimension"]

    visible = [False] * len(fig.data)
    visible[i] = True

    buttons.append(
        dict(
            label=col,
            method="update",
            args=[
                {"visible": visible},
                {
                    "title": f"Задержанное вложение ряда {col}: m={m}, tau={tau}",
                    "scene": dict(
                        xaxis_title="x(t)",
                        yaxis_title=f"x(t + {tau})",
                        zaxis_title=f"x(t + {2 * tau})",
                    ),
                },
            ],
        )
    )

first_col = columns[0]
first_tau = all_embeddings[first_col]["tau"]
first_m = all_embeddings[first_col]["dimension"]

fig.update_layout(
    title=f"Задержанное вложение ряда {first_col}: m={first_m}, tau={first_tau}",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.15,
            showactive=True,
        )
    ],
    scene=dict(
        xaxis_title="x(t)",
        yaxis_title=f"x(t + {first_tau})",
        zaxis_title=f"x(t + {2 * first_tau})",
    ),
    width=950,
    height=750,
)

fig.show()